In [1]:
import os
os.environ['OMP_NUM_THREADS'] = '1'

In [2]:
import pandas as pd
import numpy as np

In [3]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, confusion_matrix, classification_report)

---

In [5]:
# Step 1: Create Dataset

data = pd.DataFrame({

    "SoilMoisture": [
        25, 28, 30, 35,
        45, 48, 50, 52,
        60, 65, 68, 70,
        40, 42, 44, 46
    ],
    "Nitrogen": [
        40, 45, 50, 55,
        70, 75, 80, 85,
        30, 35, 40, 45,
        60, 65, 70, np.nan
    ],
    "Phosphorus": [
        30, 35, 32, 38,
        50, 55, 52, 58,
        25, 28, 30, 32,
        45, 48, 50, 52
    ],
    "Potassium": [
        35, 40, 38, 42,
        60, 65, 62, 68,
        30, 32, 35, 38,
        50, 55, 58, 60
    ],
    "Temperature": [
        22, 24, 25, 26,
        28, 30, 31, 32,
        35, 36, 38, 40,
        25, 26, 27, 28
    ],
    "Humidity": [
        75, 72, 70, 68,
        65, 62, 60, 58,
        45, 42, 40, 38,
        70, 68, 66, 64
    ],
    "Rainfall": [
        180, 190, 200, 210,
        150, 160, 170, 180,
        80, 90, 100, 110,
        140, 150, 160, 170
    ],
    "Crop": [
        "Rice", "Rice", "Rice", "Rice",
        "Wheat", "Wheat", "Wheat", "Wheat",
        "Cotton", "Cotton", "Cotton", "Cotton",
        "Maize", "Maize", "Maize", "Maize"
    ]
})

In [6]:
print("\nOriginal Dataset:\n")
data


Original Dataset:



,SoilMoisture,Nitrogen,Phosphorus,Potassium,Temperature,Humidity,Rainfall,Crop
0,25,40.0,30,35,22,75,180,Rice
1,28,45.0,35,40,24,72,190,Rice
2,30,50.0,32,38,25,70,200,Rice
3,35,55.0,38,42,26,68,210,Rice
4,45,70.0,50,60,28,65,150,Wheat
5,48,75.0,55,65,30,62,160,Wheat
6,50,80.0,52,62,31,60,170,Wheat
7,52,85.0,58,68,32,58,180,Wheat
8,60,30.0,25,30,35,45,80,Cotton
9,65,35.0,28,32,36,42,90,Cotton


---

In [7]:
# Step 2: Separate Features and Target

X = data.drop("Crop", axis=1)

y = data["Crop"]

---

In [8]:
# Step 3: Handle Missing Values

imputer = SimpleImputer(strategy="mean")

X_imputed = imputer.fit_transform(X)

---

In [9]:
# Step 4: Standardize Features

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X_imputed)

---

In [10]:
# Step 5: PCA
# Reduce 7 features → 3 components

pca = PCA(n_components=3)

X_pca = pca.fit_transform(X_scaled)

print(f"\nPCA Output Shape : {X_pca.shape}")

print(f"\nExplained Variance Ratio : {pca.explained_variance_ratio_}")


PCA Output Shape : (16, 3)

Explained Variance Ratio : [0.62682927 0.3275663  0.03656723]


---

In [11]:
# Step 6: K-Means Clustering

kmeans = KMeans(n_clusters=3, random_state=42)

clusters = kmeans.fit_predict(X_pca)

In [12]:
# Add cluster to dataset

data["AgricultureZone"] = clusters

print("\nAgricultural Zones:\n")

print(data[["Crop", "AgricultureZone"]])


Agricultural Zones:

      Crop  AgricultureZone
0     Rice                2
1     Rice                2
2     Rice                2
3     Rice                2
4    Wheat                0
5    Wheat                0
6    Wheat                0
7    Wheat                0
8   Cotton                1
9   Cotton                1
10  Cotton                1
11  Cotton                1
12   Maize                0
13   Maize                0
14   Maize                0
15   Maize                0


---

In [13]:
# Step 7: Create Final Feature Dataset

X_final = pd.DataFrame(X_pca, columns=["PCA_1", "PCA_2", "PCA_3"])

X_final["AgricultureZone"] = clusters

---

In [14]:
# Step 8: Train-Test Split

X_train, X_test, y_train, y_test = train_test_split(X_final, y, test_size=0.25, random_state=42, stratify=y)

---

In [15]:
# Step 9: Train Random Forest

model = RandomForestClassifier(n_estimators=100, random_state=42)

model = model.fit(X_train, y_train)

---